# Load Data

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# TODO: Define columns used as multivariate time-series features
TS_FEATURE_COLS = [
    "tbl_speed",
    "fom",
    "main_comp",
    "tbl_fill",
    "SREL",
    "pre_comp",
    "produced",
    "waste",
    "cyl_main",
    "cyl_pre",
    "stiffness",
    "ejection"
]


TARGET_COLS = ["total_waste", "Total impurities"]

# Load process-level targets
print("Loading Process.csv (targets)...")
process = pd.read_csv("data/Process.csv", sep=";")      

# Keep only the necessary columns
cols = ["batch", "code"] + TARGET_COLS
process = process[cols]

# Drop batches that do not have valid target values
process = process.dropna(subset=TARGET_COLS).reset_index(drop=True)
print(f"Process targets shape: {process.shape}")

# Normalize target values
target_scaler = StandardScaler()
process[TARGET_COLS] = target_scaler.fit_transform(process[TARGET_COLS])
print("Normalized target variables.")

# Load and concatenate all time-series CSV files
print("\nLoading time-series CSV files...")
dfs = []
for i in range(1, 26):
    df = pd.read_csv(f"data/Time-Series/{i}.csv", sep=";")
    dfs.append(df)
    
ts = pd.concat(dfs, ignore_index=True)
print(f"Time-series shape: {ts.shape}")

# Normalize time-series feature columns
ts_scaler = StandardScaler()
ts[TS_FEATURE_COLS] = ts_scaler.fit_transform(ts[TS_FEATURE_COLS])
print("Normalized TS features:", TS_FEATURE_COLS)

Loading Process.csv (targets)...
Process targets shape: (987, 4)
Normalized target variables.

Loading time-series CSV files...
Time-series shape: (4720208, 16)
Normalized TS features: ['tbl_speed', 'fom', 'main_comp', 'tbl_fill', 'SREL', 'pre_comp', 'produced', 'waste', 'cyl_main', 'cyl_pre', 'stiffness', 'ejection']


# Build Sequences

In [7]:
# Keep only time-series data whose batch IDs appear in the process targets
valid_batches = set(process["batch"].unique())
ts = ts[ts["batch"].isin(valid_batches)].copy()

# Fill missing feature values within each batch
def fill_group(g):
    g = g.copy()
    g = g.ffill()         # forward fill missing values
    g = g.bfill()         # backward fill any remaining gaps
    g = g.fillna(0.0)     # fill any remaining NaNs with 0.0
    return g

# Apply the filling function to each batch
ts[TS_FEATURE_COLS] = ts.groupby("batch")[TS_FEATURE_COLS].transform(fill_group)

# Compute the number of time steps for each batch
batch_lengths = ts.groupby("batch").size()

# Get the distribution of sequence lengths and its 95th percentile
p95 = np.percentile(batch_lengths.to_numpy(), 95)

# Define the maximum sequence length to control memory usage
max_seq_len = int(min(p95, 5000))
print(f"Maximum sequence length: {max_seq_len}")

# Sort batch IDs for reproducible ordering
batch_ids = sorted(batch_lengths.index)

X_list, y_list = [], []
for b in batch_ids:
    # Extract feature matrix (T, C) for each batch
    g = ts[ts["batch"] == b]
    seq = g[TS_FEATURE_COLS].values.astype(np.float32)
    
    # Truncate or pad to maximum sequence length
    if seq.shape[0] >= max_seq_len:
        seq = seq[:max_seq_len, :]
    else:
        pad_len = max_seq_len - seq.shape[0]
        pad = np.zeros((pad_len, seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])
        
    # Align each batch with its target values
    row = process.loc[process["batch"] == b, TARGET_COLS]
    y = row.iloc[0].values.astype(np.float32)
    
    X_list.append(seq)
    y_list.append(y)

X = np.stack(X_list, axis=0)    # (num_batches, T, C)
y = np.stack(y_list, axis=0)    # (num_batches, 2)

print(f"Built sequences with X shape: {X.shape}, y shape: {y.shape}")


Maximum sequence length: 5000
Built sequences with X shape: (987, 5000, 12), y shape: (987, 2)


# Train/Validation Split

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

num_batches, T, C = X.shape

# Scale input features per-channel over all timesteps and batches
X_reshaped = X.reshape(-1, C)  # Flatten: (num_batches * T, C)

x_scaler = StandardScaler()
X_scaled_reshaped = x_scaler.fit_transform(X_reshaped)

# Restore original structure: (num_batches, T, C)
X_scaled = X_scaled_reshaped.reshape(num_batches, T, C).astype(np.float32)

# Scale target variables over the 2D target array
y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y).astype(np.float32)

# Split batches into training and validation sets
X_t, X_v, y_t, y_v = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_t.shape}, y_train shape: {y_t.shape}")
print(f"X_val shape: {X_v.shape}, y_val shape: {y_v.shape}")


X_train shape: (789, 5000, 12), y_train shape: (789, 2)
X_val shape: (198, 5000, 12), y_val shape: (198, 2)


# Build and Train CNN

In [9]:
from utils.cnn import CNN_trainer

# Build and train the model
cnn_trainer = CNN_trainer(
    X_t, y_t, 
    X_v, y_v,
    epochs=100,         # Number of epochs
    batch_size=128,     # Batch size
    lr=1e-3,            # Learning rate
    l2=1e-5,            # L2 regularization coefficient
)
cnn_trainer.run()



Training Epoch 1/100
7/7 - loss: 1.2906 - MAE: 0.8743 - val_loss: 0.8977 - val_MAE: 0.7498
Learning rate: 0.001000
Validation MAE improved to 0.7498
Updated the best model

Training Epoch 2/100
7/7 - loss: 1.1385 - MAE: 0.7897 - val_loss: 0.8836 - val_MAE: 0.7481
Learning rate: 0.001000
Validation MAE improved to 0.7481
Updated the best model

Training Epoch 3/100
7/7 - loss: 0.9880 - MAE: 0.7449 - val_loss: 0.8730 - val_MAE: 0.7461
Learning rate: 0.000999
Validation MAE improved to 0.7461
Updated the best model

Training Epoch 4/100
7/7 - loss: 0.9008 - MAE: 0.7176 - val_loss: 0.8561 - val_MAE: 0.7381
Learning rate: 0.000998
Validation MAE improved to 0.7381
Updated the best model

Training Epoch 5/100
7/7 - loss: 0.8622 - MAE: 0.7155 - val_loss: 0.8375 - val_MAE: 0.7285
Learning rate: 0.000996
Validation MAE improved to 0.7285
Updated the best model

Training Epoch 6/100
7/7 - loss: 0.8663 - MAE: 0.6923 - val_loss: 0.8302 - val_MAE: 0.7240
Learning rate: 0.000994
Validation MAE impr

# Build and Train LSTM

In [10]:
import importlib
import utils.lstm as lstm
importlib.reload(lstm)

from utils.lstm import LSTM_trainer

lstm_trainer = LSTM_trainer(
    X_t, y_t, 
    X_v, y_v, 
    epochs=100, 
    batch_size=128, 
    lr=1e-3,
    l2=1e-5,
)
lstm_trainer.run()


Training Epoch 1/100
7/7 - loss: 1.0469 - MAE: 0.7504 - val_loss: 0.9135 - val_MAE: 0.7496
Learning rate: 0.001000
Validation MAE improved to 0.7496
Updated the best LSTM model

Training Epoch 2/100
7/7 - loss: 1.0298 - MAE: 0.7524 - val_loss: 0.8970 - val_MAE: 0.7459
Learning rate: 0.001000
Validation MAE improved to 0.7459
Updated the best LSTM model

Training Epoch 3/100
7/7 - loss: 0.9863 - MAE: 0.7513 - val_loss: 0.8835 - val_MAE: 0.7436
Learning rate: 0.000999
Validation MAE improved to 0.7436
Updated the best LSTM model

Training Epoch 4/100
7/7 - loss: 1.1636 - MAE: 0.7416 - val_loss: 0.8699 - val_MAE: 0.7361
Learning rate: 0.000998
Validation MAE improved to 0.7361
Updated the best LSTM model

Training Epoch 5/100
7/7 - loss: 0.9735 - MAE: 0.7373 - val_loss: 0.8511 - val_MAE: 0.7292
Learning rate: 0.000996
Validation MAE improved to 0.7292
Updated the best LSTM model

Training Epoch 6/100
7/7 - loss: 0.9197 - MAE: 0.7347 - val_loss: 0.8313 - val_MAE: 0.7218
Learning rate: 0.0

# Save the Best Model